<a href="https://colab.research.google.com/github/zombimann/Mathematical-video-animations-and-visualization/blob/main/_Kids_Middle_School_Slopes_Rise_Over_Run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Understanding Slope through Stair Steepness

This notebook contains the code to generate an educational animation designed for middle school students. The visualization uses the physical metaphor of climbing stairs to introduce the mathematical concept of slope.

## Learning Objectives

By the end of this lesson/video, students should be able to:
1. Define slope as a measure of steepness.
2. Identify the horizontal change as 'Run'.
3. Identify the vertical change as 'Rise'.
4. Understand that slope is calculated as the ratio of Rise over Run (m = rise/run).
5. Compare different slopes and understand that a larger numerical value represents a steeper incline.

## Key Concepts Covered

### 1. The Physical Metaphor
Stairs provide a concrete example of slope. A shallow staircase is easy to climb (gentle slope), while a tall, narrow staircase requires more effort (steep slope).

### 2. Rise over Run
* **Rise**: The vertical distance between steps (change in y).
* **Run**: The horizontal distance of the step tread (change in x).
* **The Formula**: Slope (m) = Rise / Run.

### 3. Comparing Magnitudes
* **Gentle Slope**: Example shown with Rise=1, Run=2. Calculation: 1/2 = 0.5.
* **Steep Slope**: Example shown with Rise=3, Run=1. Calculation: 3/1 = 3.
* **Comparison**: Since 3 > 0.5, the second staircase is mathematically steeper.

## Guide for Educators and Parents

* **Pause and Reflect**: Encourage students to pause the video during the 'Which is steeper?' prompt to make their own predictions before the math is revealed.
* **Real-world Application**: Ask students to look at ramps, hills, or roofs in their neighborhood and estimate if they have a 'large' or 'small' slope based on the rise and run.
* **Variable Modification**: Advanced students can modify the `GENTLE` and `STEEP` dictionaries in the code below to see how changing the numbers affects the generated animation.

In [5]:

"""
Author: Mugambi Ndwiga
Instagram: @craftsandengineering
GitHub: github.com/zombimann/Mathematical-video-animations-and-visualization
Level: Middle School
Concept: Slope as Stair Steepness - Rise Over Run
"""

import os
import warnings

import imageio
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import Arc
import numpy as np
from PIL import Image

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# PARAMS
# ─────────────────────────────────────────────────────────────────────────────

VIDEO = dict(
    width_px=1080,
    height_px=1080,
    fps=24,
    dpi=108,
    output="slope_rise_over_run_clean_v5.mp4",
)

PAL = dict(
    bg="#FFFFFF",
    grid="#E6E6E6",
    axis="#4A4A4A",
    navy="#2A3D66",
    sage="#86A788",
    coral="#E5989B",
    mustard="#E9C46A",
    run="#2A9D8F",
    rise="#E76F51",
    dark="#1A1A2E",
    steep="#C0404A",
    text="#23304A",
    white="#FFFFFF",
    panel="#F7F9FC",
    panel_line="#D7DFEA",
    closing="#2A3D66",
    water="#2A3D66",
)

FONT = dict(
    family="Liberation Sans",
    title=44,
    heading=34,
    body=28,
    label=24,
    small=18,
    watermark=12,
)

GENTLE = dict(run=2, rise=1)
STEEP = dict(run=1, rise=3)

T = dict(
    hook=5.0,
    gentle=8.0,
    steep=8.0,
    person=7.0,
    recap=5.5,
    closing=2.0,
)

WATERMARK = "© Mugambi Ndwiga / @craftsandengineering"

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def smooth(x):
    x = max(0.0, min(1.0, float(x)))
    return x * x * (3.0 - 2.0 * x)

def clamp(x, a=0.0, b=1.0):
    return max(a, min(b, x))

def make_fig():
    w = VIDEO["width_px"] / VIDEO["dpi"]
    h = VIDEO["height_px"] / VIDEO["dpi"]
    fig = plt.figure(figsize=(w, h), facecolor=PAL["bg"])
    return fig

def to_rgb(fig):
    fig.canvas.draw()
    buf = fig.canvas.buffer_rgba()
    img = np.asarray(buf)[:, :, :3].copy()
    if img.shape[1] != VIDEO["width_px"] or img.shape[0] != VIDEO["height_px"]:
        img = np.array(Image.fromarray(img).resize((VIDEO["width_px"], VIDEO["height_px"]), Image.LANCZOS))
    return img

def add_axes(fig, rect=(0.08, 0.10, 0.84, 0.79)):
    ax = fig.add_axes(rect)
    ax.set_facecolor(PAL["bg"])
    ax.set_xlim(-0.4, 8.6)
    ax.set_ylim(-0.7, 7.1)
    ax.set_aspect("equal")
    ax.axis("off")

    # Light grid
    for x in range(0, 9):
        ax.axvline(x, color=PAL["grid"], lw=0.9, zorder=0)
    for y in range(0, 8):
        ax.axhline(y, color=PAL["grid"], lw=0.9, zorder=0)

    # Axes and arrows
    ax.axhline(0, color=PAL["axis"], lw=1.9, zorder=1)
    ax.axvline(0, color=PAL["axis"], lw=1.9, zorder=1)
    ax.annotate("", xy=(8.55, 0), xytext=(8.38, 0),
                arrowprops=dict(arrowstyle="->", color=PAL["axis"], lw=1.9), zorder=2)
    ax.annotate("", xy=(0, 7.05), xytext=(0, 6.88),
                arrowprops=dict(arrowstyle="->", color=PAL["axis"], lw=1.9), zorder=2)

    ax.text(8.42, -0.39, "x", fontsize=FONT["body"], color=PAL["axis"],
            fontfamily=FONT["family"], style="italic", ha="center", va="center", zorder=3)
    ax.text(-0.30, 6.95, "y", fontsize=FONT["body"], color=PAL["axis"],
            fontfamily=FONT["family"], style="italic", ha="center", va="center", zorder=3)

    for v in range(2, 8, 2):
        ax.text(v, -0.27, str(v), fontsize=FONT["small"], color="#A7A7A7",
                ha="center", va="top", fontfamily=FONT["family"], zorder=3)
        ax.text(-0.21, v, str(v), fontsize=FONT["small"], color="#A7A7A7",
                ha="right", va="center", fontfamily=FONT["family"], zorder=3)
    return ax

def add_title(fig, text, color=PAL["navy"], y=0.965, fs=None, alpha=1.0):
    if fs is None:
        fs = FONT["heading"]
    fig.text(0.5, y, text, ha="center", va="top",
             fontsize=fs, fontfamily=FONT["family"],
             color=color, fontweight="bold", alpha=alpha)

def add_subtitle(fig, text, color=PAL["text"], y=0.915, fs=None, alpha=1.0):
    if fs is None:
        fs = FONT["body"]
    fig.text(0.5, y, text, ha="center", va="top",
             fontsize=fs, fontfamily=FONT["family"],
             color=color, fontweight="bold", alpha=alpha)

def add_grade_tag(fig, alpha=1.0):
    fig.text(0.985, 0.985, "For Kids: Middle School", ha="right", va="top",
             fontsize=FONT["small"]-2, fontfamily=FONT["family"], color=PAL["white"],
             fontweight="bold", alpha=alpha,
             bbox=dict(boxstyle="round,pad=0.26", facecolor=PAL["navy"] + "CC", edgecolor="none"))

def add_watermark(fig):
    fig.text(0.993, 0.006, WATERMARK, ha="right", va="bottom",
             fontsize=FONT["watermark"], fontfamily=FONT["family"],
             color=PAL["water"], alpha=0.40, style="italic")

def draw_card(fig, x, y, w, h, text, color=PAL["navy"], fs=None, alpha=1.0,
              face="white", edge=None, lw=2.0):
    if fs is None:
        fs = FONT["body"]
    if edge is None:
        edge = color
    ax = fig.add_axes([x, y, w, h])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.add_patch(patches.FancyBboxPatch(
        (0, 0), 1, 1,
        boxstyle="round,pad=0.02,rounding_size=0.04",
        facecolor=face, edgecolor=edge, linewidth=lw, alpha=alpha
    ))
    ax.text(0.5, 0.5, text, ha="center", va="center",
            fontsize=fs, fontfamily=FONT["family"], color=color,
            fontweight="bold", alpha=alpha, wrap=True)
    return ax

def draw_staircase(ax, ox, oy, run, rise, n, fill, border, alpha=1.0, z=4):
    x, y = ox, oy
    for _ in range(n):
        ax.add_patch(patches.Rectangle(
            (x, y), run, rise, lw=1.8, edgecolor=border,
            facecolor=fill, alpha=0.45 * alpha, zorder=z))
        ax.plot([x, x + run], [y, y], color=border, lw=1.8, alpha=alpha, zorder=z + 1)
        ax.plot([x + run, x + run], [y, y + rise], color=border, lw=1.8, alpha=alpha, zorder=z + 1)
        x += run
        y += rise

def draw_line(ax, ox, oy, run, rise, n, color, lw=4.0, progress=1.0, alpha=1.0, z=6):
    x1 = ox + n * run * progress
    y1 = oy + n * rise * progress
    ax.plot([ox, x1], [oy, y1], color=color, lw=lw, alpha=alpha, solid_capstyle="round", zorder=z)

def draw_rise(ax, ox, oy, rise, color=PAL["rise"], alpha=1.0):
    x = ox
    ax.annotate("", xy=(x, oy + rise), xytext=(x, oy),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.4), alpha=alpha, zorder=8)
    ax.text(x - 0.42, oy + rise * 0.52, "rise", fontsize=FONT["body"],
            fontfamily=FONT["family"], color=color, fontweight="bold",
            ha="center", va="center", alpha=alpha, zorder=9,
            bbox=dict(boxstyle="round,pad=0.24", facecolor="white", edgecolor="none", alpha=0.92))

def draw_run(ax, ox, oy, run, color=PAL["run"], alpha=1.0):
    y = oy - 0.34
    ax.annotate("", xy=(ox + run, y), xytext=(ox, y),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.4), alpha=alpha, zorder=8)
    ax.text(ox + run * 0.5, y - 0.26, "run", fontsize=FONT["body"],
            fontfamily=FONT["family"], color=color, fontweight="bold",
            ha="center", va="top", alpha=alpha, zorder=9,
            bbox=dict(boxstyle="round,pad=0.24", facecolor="white", edgecolor="none", alpha=0.92))

def draw_eq_card(fig, x, y, w, h, lines, color=PAL["navy"], alpha=1.0):
    text = "\n".join(lines)
    draw_card(fig, x, y, w, h, text, color=color, fs=FONT["heading"]-1,
              alpha=alpha, face="white", edge=color, lw=2.1)

def draw_person(ax, cx, cy, scale=0.42, color=None, phase=0.0, alpha=1.0, z=10):
    if color is None:
        color = "#7A4F2C"
    lw = 2.6
    swing = 0.20 * scale * np.sin(phase * 2 * np.pi)
    ax.add_patch(plt.Circle((cx, cy + 0.53 * scale), 0.17 * scale,
                            color=PAL["mustard"], ec=color, lw=lw, alpha=alpha, zorder=z))
    ax.plot([cx, cx], [cy, cy + 0.36 * scale], color=color, lw=lw * 2,
            solid_capstyle="round", alpha=alpha, zorder=z)
    ax.plot([cx - 0.18 * scale, cx, cx + 0.18 * scale],
            [cy + 0.18 * scale, cy + 0.28 * scale, cy + 0.16 * scale],
            color=color, lw=lw * 1.4, solid_capstyle="round", alpha=alpha, zorder=z)
    ax.plot([cx, cx + swing], [cy, cy - 0.35 * scale],
            color=color, lw=lw * 1.4, solid_capstyle="round", alpha=alpha, zorder=z)
    ax.plot([cx, cx - swing], [cy, cy - 0.35 * scale],
            color=color, lw=lw * 1.4, solid_capstyle="round", alpha=alpha, zorder=z)

def draw_vs(fig, x, y, alpha=1.0):
    fig.text(x, y, "vs", ha="center", va="center",
             fontsize=FONT["title"], fontfamily=FONT["family"],
             color=PAL["text"], fontweight="bold", alpha=alpha)

# ─────────────────────────────────────────────────────────────────────────────
# SCENES
# ─────────────────────────────────────────────────────────────────────────────

def scene_hook(t, dur):
    fig = make_fig()
    ax = add_axes(fig)

    f = smooth(t / 1.2)

    draw_staircase(ax, 0.9, 0.55, GENTLE["run"], GENTLE["rise"], 3,
                   PAL["sage"], "#4F6B57", alpha=f)
    draw_staircase(ax, 5.0, 0.55, STEEP["run"], STEEP["rise"], 2,
                   PAL["coral"], "#943040", alpha=f)
    ax.text(2.1, 0.05, "A", fontsize=FONT["title"], fontfamily=FONT["family"],
            color=PAL["sage"], fontweight="bold", ha="center", va="center", alpha=f)
    ax.text(6.1, 0.05, "B", fontsize=FONT["title"], fontfamily=FONT["family"],
            color=PAL["steep"], fontweight="bold", ha="center", va="center", alpha=f)

    add_title(fig, "Which staircase is steeper?", color=PAL["navy"], fs=FONT["heading"] + 2, alpha=f)
    add_subtitle(fig, "Slope is the math word for steepness.", color=PAL["text"], fs=FONT["body"], alpha=f)
    add_grade_tag(fig, alpha=f)
    add_watermark(fig)

    rgb = to_rgb(fig)
    plt.close(fig)
    return rgb

def scene_gentle(t, dur):
    fig = make_fig()
    ax = add_axes(fig)

    p1 = smooth(t / 1.6)
    p2 = smooth((t - 1.7) / 1.4)
    p3 = smooth((t - 3.3) / 1.2)
    p4 = smooth((t - 4.9) / 1.0)

    draw_staircase(ax, 1.1, 0.55, GENTLE["run"], GENTLE["rise"], 3,
                   PAL["sage"], "#4F6B57", alpha=0.18 + 0.82 * p1)
    draw_line(ax, 1.1, 0.55, GENTLE["run"], GENTLE["rise"], 3,
              PAL["navy"], lw=4.2, progress=p1, alpha=0.25 + 0.65 * p1)

    if p2 > 0:
        draw_rise(ax, 1.1 + GENTLE["run"], 0.55, GENTLE["rise"], color=PAL["rise"], alpha=p2)
    if p3 > 0:
        draw_run(ax, 1.1, 0.55, GENTLE["run"], color=PAL["run"], alpha=p3)

    if p4 > 0:
        draw_eq_card(fig, 0.31, 0.78, 0.38, 0.13,
                     ["m = rise / run", "1 / 2 = 0.5"], color=PAL["navy"], alpha=p4)

    add_title(fig, "Gentle slope", color=PAL["navy"], fs=FONT["heading"], alpha=1.0)
    add_grade_tag(fig)
    add_watermark(fig)

    rgb = to_rgb(fig)
    plt.close(fig)
    return rgb

def scene_steep(t, dur):
    fig = make_fig()
    ax = add_axes(fig)

    p1 = smooth(t / 1.6)
    p2 = smooth((t - 1.7) / 1.4)
    p3 = smooth((t - 3.3) / 1.2)
    p4 = smooth((t - 4.9) / 1.0)

    draw_staircase(ax, 4.9, 0.55, STEEP["run"], STEEP["rise"], 2,
                   PAL["coral"], "#943040", alpha=0.18 + 0.82 * p1)
    draw_line(ax, 4.9, 0.55, STEEP["run"], STEEP["rise"], 2,
              PAL["steep"], lw=4.4, progress=p1, alpha=0.25 + 0.65 * p1)

    if p2 > 0:
        draw_rise(ax, 4.9 + STEEP["run"], 0.55, STEEP["rise"], color=PAL["rise"], alpha=p2)
    if p3 > 0:
        draw_run(ax, 4.9, 0.55, STEEP["run"], color=PAL["run"], alpha=p3)

    if p4 > 0:
        draw_eq_card(fig, 0.31, 0.78, 0.38, 0.13,
                     ["m = rise / run", "3 / 1 = 3"], color=PAL["steep"], alpha=p4)

    add_title(fig, "Steep slope", color=PAL["steep"], fs=FONT["heading"], alpha=1.0)
    add_grade_tag(fig)
    add_watermark(fig)

    rgb = to_rgb(fig)
    plt.close(fig)
    return rgb

def scene_person(t, dur):
    fig = make_fig()
    ax = add_axes(fig)

    p = smooth(t / 1.0)
    split = dur / 2.0

    # gentle on left, steep on right as quiet context
    draw_staircase(ax, 0.7, 0.55, GENTLE["run"], GENTLE["rise"], 3,
                   PAL["sage"], "#4F6B57", alpha=0.33)
    draw_staircase(ax, 5.0, 0.55, STEEP["run"], STEEP["rise"], 2,
                   PAL["coral"], "#943040", alpha=0.33)
    draw_line(ax, 0.7, 0.55, GENTLE["run"], GENTLE["rise"], 3, PAL["navy"], lw=4.0, alpha=0.35)
    draw_line(ax, 5.0, 0.55, STEEP["run"], STEEP["rise"], 2, PAL["steep"], lw=4.0, alpha=0.35)

    if t <= split:
        q = smooth(t / split)
        step = min(2, int(q * 3))
        frac = (q * 3) % 1.0
        cx = 0.7 + step * GENTLE["run"] + frac * GENTLE["run"] * 0.9 + 0.24
        cy = 0.55 + step * GENTLE["rise"] + frac * GENTLE["rise"] * 0.25 + 0.10
        draw_person(ax, cx, cy, scale=0.45, phase=t * 1.5, alpha=p)
        add_title(fig, "Same climber, easier path", color=PAL["navy"], fs=FONT["heading"], alpha=1.0)
    else:
        q = smooth((t - split) / split)
        step = min(1, int(q * 2))
        frac = (q * 2) % 1.0
        cx = 5.0 + step * STEEP["run"] + frac * STEEP["run"] * 0.65 + 0.17
        cy = 0.55 + step * STEEP["rise"] + frac * STEEP["rise"] * 0.20 + 0.10
        draw_person(ax, cx, cy, scale=0.45, phase=t * 1.7, color="#943040", alpha=p)
        add_title(fig, "Same climber, harder path", color=PAL["steep"], fs=FONT["heading"], alpha=1.0)

    add_grade_tag(fig)
    add_watermark(fig)

    rgb = to_rgb(fig)
    plt.close(fig)
    return rgb

def scene_recap(t, dur):
    fig = make_fig()
    ax = add_axes(fig)

    a1 = smooth(t / 0.8)
    a2 = smooth((t - 0.8) / 0.8)
    a3 = smooth((t - 1.6) / 0.8)

    draw_card(fig, 0.12, 0.62, 0.32, 0.20, "m = 0.5", color=PAL["navy"], fs=FONT["title"]-2, alpha=a1)
    draw_card(fig, 0.56, 0.62, 0.32, 0.20, "m = 3", color=PAL["steep"], fs=FONT["title"]-2, alpha=a2)

    if a3 > 0:
        draw_card(fig, 0.28, 0.38, 0.44, 0.15, "Bigger m = steeper line", color=PAL["navy"], fs=FONT["heading"]-1, alpha=a3,
                  face="#FFFDF7", edge=PAL["mustard"], lw=2.2)

    add_title(fig, "Rise over run", color=PAL["navy"], fs=FONT["heading"], alpha=1.0)
    add_subtitle(fig, "That is how you read slope.", color=PAL["text"], fs=FONT["body"], alpha=a3)
    add_grade_tag(fig)
    add_watermark(fig)

    rgb = to_rgb(fig)
    plt.close(fig)
    return rgb

def scene_closing(t, dur):
    fade_in = smooth(t / 0.45)
    fade_out = smooth((dur - t) / 0.45)
    a = min(fade_in, fade_out)

    fig = make_fig()
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_facecolor(PAL["closing"])

    ax.text(0.5, 0.60, "Made by Mugambi Ndwiga", ha="center", va="center",
            fontsize=FONT["title"], fontfamily=FONT["family"], color="white",
            fontweight="bold", alpha=a)
    ax.text(0.5, 0.45, "@craftsandengineering", ha="center", va="center",
            fontsize=FONT["heading"], fontfamily=FONT["family"], color="white",
            fontweight="bold", alpha=a)
    ax.text(0.995, 0.006, WATERMARK, ha="right", va="bottom",
            fontsize=FONT["watermark"], fontfamily=FONT["family"],
            color="white", alpha=0.35, style="italic")

    rgb = to_rgb(fig)
    plt.close(fig)
    return rgb

# ─────────────────────────────────────────────────────────────────────────────
# VIDEO
# ─────────────────────────────────────────────────────────────────────────────

def frames(fn, dur, fps):
    n = max(1, int(round(dur * fps)))
    return [fn(i / fps, dur) for i in range(n)]

def build_video():
    fps = VIDEO["fps"]
    scenes = [
        ("Hook", scene_hook, T["hook"]),
        ("Gentle", scene_gentle, T["gentle"]),
        ("Steep", scene_steep, T["steep"]),
        ("Person", scene_person, T["person"]),
        ("Recap", scene_recap, T["recap"]),
        ("Closing", scene_closing, T["closing"]),
    ]

    all_frames = []
    for name, fn, dur in scenes:
        print(f"Rendering: {name} ({dur:.1f}s)")
        all_frames.extend(frames(fn, dur, fps))

    out = VIDEO["output"]
    os.makedirs(os.path.dirname(os.path.abspath(out)) or ".", exist_ok=True)
    print(f"Writing: {out}")
    with imageio.get_writer(
        out, fps=fps, codec="libx264", quality=8, macro_block_size=1,
        ffmpeg_params=["-crf", "26", "-preset", "slow", "-pix_fmt", "yuv420p"]
    ) as w:
        for fr in all_frames:
            w.append_data(fr)

    size_mb = os.path.getsize(out) / 1e6
    print(f"Done: {size_mb:.2f} MB")
    return out

def display_and_download(path=None):
    if path is None:
        path = VIDEO["output"]
    path = os.path.abspath(path)
    if not os.path.exists(path):
        print(f"Missing file: {path}")
        return

    size_mb = os.path.getsize(path) / 1e6
    print(f"Video ready: {path} ({size_mb:.2f} MB)")

    try:
        from IPython.display import display, Video, HTML
        import base64

        display(Video(path, embed=True, width=720, html_attributes="controls autoplay loop"))
        with open(path, "rb") as f:
            b64 = base64.b64encode(f.read()).decode()
        fn = os.path.basename(path)
        display(HTML(
            f'<a href="data:video/mp4;base64,{b64}" download="{fn}" '
            f'style="display:inline-block;padding:11px 24px;background:#2A3D66;'
            f'color:#fff;border-radius:8px;text-decoration:none;'
            f'font-family:sans-serif;font-size:16px;font-weight:bold;">'
            f'Download {fn}</a>'
        ))
    except Exception:
        print(path)

if __name__ == "__main__":
    build_video()
    try:
        get_ipython  # noqa: F821
        display_and_download()
    except NameError:
        pass


Rendering: Hook (5.0s)
Rendering: Gentle (8.0s)
Rendering: Steep (8.0s)
Rendering: Person (7.0s)
Rendering: Recap (5.5s)
Rendering: Closing (2.0s)
Writing: slope_rise_over_run_clean_v5.mp4
Done: 0.35 MB
Video ready: /content/slope_rise_over_run_clean_v5.mp4 (0.35 MB)
